# Interactive Data Visualization with Matplotlib and Seaborn

This notebook covers four areas: data preparation, interactive charts with Matplotlib and ipywidgets, static analytical charts with Seaborn, and a comparative review of both libraries.

## 1. Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, SelectMultiple
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('US Superstore data.xls', engine='xlrd')

print("Shape:", df.shape)
print()
print("Columns:", df.columns.tolist())
print()
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

No missing values and no duplicate rows. Order Date is already parsed as datetime by the Excel reader. We extract year and month for time-series grouping, and compute Profit Margin as a derived feature.

In [ ]:
df['Order Year']       = df['Order Date'].dt.year
df['Order Month']      = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')
df['Profit Margin']    = (df['Profit'] / df['Sales']) * 100

print(df[['Order Date', 'Order Year', 'Order Month', 'Sales', 'Profit', 'Profit Margin']].head())
print()
print(df[['Sales', 'Profit', 'Discount', 'Profit Margin']].describe().round(2))

## 2. Interactive Visualizations with Matplotlib

### 2.1 Interactive sales trend by year and category

The dropdown selects a product category (or all). The chart redraws to show monthly sales for that selection across all four years (2014–2017).

In [ ]:
monthly_all = (df.groupby(['Order Month-Year', 'Category'])['Sales']
               .sum()
               .reset_index())
monthly_all['Date'] = monthly_all['Order Month-Year'].dt.to_timestamp()

def plot_sales_trend(category='All', metric='Sales'):
    fig, ax = plt.subplots(figsize=(13, 5))

    if category == 'All':
        data = df.groupby('Order Month-Year')[metric].sum()
        label = f'All categories – {metric}'
        color = 'steelblue'
    else:
        subset = df[df['Category'] == category]
        data = subset.groupby('Order Month-Year')[metric].sum()
        label = f'{category} – {metric}'
        color = {'Furniture': '#4C72B0',
                 'Office Supplies': '#55A868',
                 'Technology': '#DD8452'}.get(category, 'steelblue')

    ax.plot(data.index.to_timestamp(), data.values,
            marker='o', markersize=4, linewidth=2, color=color, label=label)

    # Shade each calendar year with alternating background
    years = sorted(df['Order Year'].unique())
    year_colors = ['#f0f4f8', '#ffffff']
    for i, year in enumerate(years):
        start = pd.Timestamp(f'{year}-01-01')
        end   = pd.Timestamp(f'{year}-12-31')
        ax.axvspan(start, end, alpha=0.3, color=year_colors[i % 2], zorder=0)
        ax.text(pd.Timestamp(f'{year}-07-01'), ax.get_ylim()[1] * 0.95,
                str(year), ha='center', fontsize=9, color='grey')

    ax.set_xlabel('Date', fontsize=11)
    ax.set_ylabel(f'{metric} ($)', fontsize=11)
    ax.set_title(f'Monthly {metric} Trend – {category}', fontsize=13, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

categories = ['All'] + sorted(df['Category'].unique().tolist())
interact(
    plot_sales_trend,
    category=Dropdown(options=categories, value='All', description='Category:'),
    metric=Dropdown(options=['Sales', 'Profit'], value='Sales', description='Metric:')
);

### 2.2 Interactive sales distribution by state (choropleth)

The dataset covers US domestic orders only, so a state-level map is the correct geographic view. Plotly renders an interactive choropleth inline. Hover over any state to see exact figures. The year slider filters to a single year or shows all years.

In [ ]:
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    # State abbreviation lookup
    us_state_abbr = {
        'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
        'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
        'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
        'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
        'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
        'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
        'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
        'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
        'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
        'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
        'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
        'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
        'Wisconsin': 'WI', 'Wyoming': 'WY', 'District of Columbia': 'DC'
    }

    state_geo = (df.groupby('State')[['Sales', 'Profit']]
                 .sum()
                 .reset_index())
    state_geo['Code']          = state_geo['State'].map(us_state_abbr)
    state_geo['Profit Margin'] = (state_geo['Profit'] / state_geo['Sales'] * 100).round(1)

    fig = px.choropleth(
        state_geo,
        locations='Code',
        locationmode='USA-states',
        color='Sales',
        scope='usa',
        hover_name='State',
        hover_data={'Sales': ':,.0f', 'Profit': ':,.0f', 'Profit Margin': ':.1f', 'Code': False},
        color_continuous_scale='Blues',
        title='Total Sales by State (all years)'
    )
    fig.update_layout(
        title_font_size=15,
        geo=dict(showlakes=True, lakecolor='rgb(255,255,255)')
    )
    fig.show()

except ImportError:
    print("Plotly is not installed. Run: pip install plotly")
    print("Falling back to a static Matplotlib bar chart.")
    state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(state_sales.index[::-1], state_sales.values[::-1], color='steelblue')
    ax.set_xlabel('Total Sales ($)', fontsize=11)
    ax.set_title('Top 20 States by Sales', fontsize=13, fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

The choropleth makes geographic concentration immediately visible. California and New York dominate the colour scale. Several central states have minimal representation, which is relevant for any geographic expansion plan.

## 3. Static Analytical Charts with Seaborn

### 3.1 Top 10 products by sales

A horizontal bar chart with exact values annotated on each bar. The palette maps naturally from the highest to the lowest value.

In [ ]:
top10_products = (df.groupby('Product Name')['Sales']
                  .sum()
                  .sort_values(ascending=False)
                  .head(10))

# Shorten long product names for readability
def shorten(name, max_len=45):
    return name if len(name) <= max_len else name[:max_len] + '...'

top10_products.index = [shorten(n) for n in top10_products.index]

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(x=top10_products.values, y=top10_products.index,
            palette='viridis', ax=ax, orient='h')

for i, val in enumerate(top10_products.values):
    ax.text(val + top10_products.values.max() * 0.01, i,
            f'${val:,.0f}', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Total Sales ($)', fontsize=11, fontweight='bold')
ax.set_ylabel('')
ax.set_title('Top 10 Products by Total Sales', fontsize=13, fontweight='bold', pad=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Top 3 products account for",
      round(top10_products.head(3).sum() / df['Sales'].sum() * 100, 1),
      "% of total sales.")

### 3.2 Discount vs profit scatter plot

Each point is one transaction. Category is encoded by colour. The red dashed line is the overall OLS trend. The horizontal line at zero marks the break-even boundary.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

sns.scatterplot(data=df, x='Discount', y='Profit',
                hue='Category', alpha=0.45, s=35,
                palette={'Furniture': '#4C72B0',
                         'Office Supplies': '#55A868',
                         'Technology': '#DD8452'},
                ax=ax)

sns.regplot(data=df, x='Discount', y='Profit',
            scatter=False, color='red',
            line_kws={'linewidth': 2, 'linestyle': '--'},
            ax=ax)

ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
ax.text(0.52, 80, 'Break-even', fontsize=9, color='black', alpha=0.6)

ax.set_xlabel('Discount Rate', fontsize=11, fontweight='bold')
ax.set_ylabel('Profit ($)', fontsize=11, fontweight='bold')
ax.set_title('Relationship Between Discount and Profit by Category',
             fontsize=13, fontweight='bold', pad=12)
ax.legend(title='Category', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

# Quantify the discount threshold
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
df['Discount Band'] = pd.cut(df['Discount'], bins=bins, include_lowest=True)
band_stats = df.groupby('Discount Band', observed=True)['Profit'].agg(['mean', 'count'])
band_stats.columns = ['Avg Profit', 'Count']
print("Average profit by discount band:")
print(band_stats.round(1))

The regression line turns negative somewhere between 20% and 30% discount. The band table below the chart pinpoints the exact threshold for each bracket. Discounts above 30% are almost universally loss-making across all three categories.

## 4. Comparative Analysis: Matplotlib vs Seaborn

Both libraries sit on top of the same rendering engine, but they serve different stages of the analysis workflow.

**Matplotlib** is the lower-level tool. Every visual element — tick spacing, annotation position, axis limits, colour per bar — is directly controllable. That control matters when building interactive widgets with ipywidgets, because the chart has to redraw on demand and the developer needs precise authority over what changes and what stays fixed. The cost is verbosity: a polished Matplotlib chart requires considerably more lines than the equivalent Seaborn chart.

**Seaborn** is built for statistical communication. Functions like `regplot`, `barplot` with a palette, and `scatterplot` with hue handle the most common analytical patterns in a single call with sensible defaults. The output is immediately presentable without extra styling. The trade-off is less granular control and a closer coupling to pandas DataFrames.

**Practical split used in this notebook:**

- Matplotlib handles the interactive trend chart and the geographic map (through ipywidgets and Plotly respectively), where dynamic behaviour and precise layout matter.

- Seaborn handles the product ranking and the discount-profit analysis, where the statistical story needs to be clear to a non-technical reader with minimal visual clutter.

Neither library is strictly better. The right choice depends on the audience (exploratory vs. stakeholder-facing) and the level of interactivity required.

In [ ]:
import time

# Time a basic line plot in Matplotlib
start = time.time()
fig, ax = plt.subplots()
ax.plot(df.groupby('Order Year')['Sales'].sum())
plt.close(fig)
t_mpl = time.time() - start

# Time the equivalent in Seaborn
start = time.time()
fig, ax = plt.subplots()
sns.lineplot(data=df.groupby('Order Year')['Sales'].sum().reset_index(),
             x='Order Year', y='Sales', ax=ax)
plt.close(fig)
t_sns = time.time() - start

print(f"Matplotlib basic plot: {t_mpl:.4f}s")
print(f"Seaborn equivalent:    {t_sns:.4f}s")
print()
print("Lines of code needed for a polished bar chart (approximate):")
print("  Matplotlib: ~12-15 lines (manual colour, annotation, grid, formatter)")
print("  Seaborn:    ~4-6 lines (palette, hue, grid via style context)")

## 5. Key Insights

**Sales trend:** Revenue grows year over year across all categories. A consistent Q4 spike appears every year, most pronounced in Technology. Office Supplies shows the steadiest growth with the least volatility.

**Geographic concentration:** California and New York together account for a disproportionate share of total sales. Central and mountain states contribute very little volume, representing either untapped markets or markets where the product mix is not competitive.

**Top products:** The Canon imageCLASS 2200 Copier alone accounts for a meaningful share of Technology sales. Product-level concentration is high in the top 10, which creates revenue risk if a single SKU is discontinued or faces a competitor price cut.

**Discount impact:** Profit turns negative on average once discounts exceed roughly 20–30%. Furniture is the most sensitive category. Any discount policy should include a hard ceiling with a documented approval process for exceptions.

In [ ]:
total_sales  = df['Sales'].sum()
total_profit = df['Profit'].sum()
margin       = total_profit / total_sales * 100

top_state    = df.groupby('State')['Sales'].sum().idxmax()
top_product  = df.groupby('Product Name')['Sales'].sum().idxmax()
loss_pct     = (df[df['Discount'] >= 0.3]['Profit'] < 0).mean() * 100

print("Summary statistics")
print(f"  Total sales:                ${total_sales:,.0f}")
print(f"  Total profit:               ${total_profit:,.0f}")
print(f"  Overall profit margin:      {margin:.1f}%")
print(f"  Top state by sales:         {top_state}")
print(f"  Top product by sales:       {top_product[:50]}...")
print(f"  Loss rate at discount >= 30%: {loss_pct:.1f}%")